In [1]:
%%bash
git clone https://github.com/yogesh1801/mistral-hackathon.git

Cloning into 'mistral-hackathon'...


In [2]:
cd mistral-hackathon/

/home/jovyan/mistral-hackathon


/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
pip install -r requirements.txt-qq


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt-qq'
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install wandb -qq

Note: you may need to restart the kernel to use updated packages.


In [5]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
# Install transformers branch for Ministral
!pip install git+https://github.com/huggingface/transformers.git@bf3f0ae70d0e902efab4b8517fce88f6697636ce
!pip install --no-deps trl==0.22.2

In [6]:
from unsloth import FastVisionModel
import torch
max_seq_length = 2048
lora_rank = 32
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/Ministral-3-3B-Instruct-2512",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    fast_inference = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'max_position_embeddings'}


==((====))==  Unsloth 2026.2.1: Fast Ministral3 patching. Transformers: 5.0.0.dev0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'max_position_embeddings'}


Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

In [7]:
model = FastVisionModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model.vision_tower.transformer` require gradients


In [8]:
from unsloth import get_chat_template
# Apply the standard ChatML template to your tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml" # You can also use "mistral" depending on your base model
)


import wandb



from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    temperature = 0.7,
    learning_rate = 5e-5,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations = 4,
    max_prompt_length = 1024,
    max_completion_length = 128,
    num_train_epochs = 1, # Set to 1 for a full training run
    save_steps = 10,
    report_to = "wandb", # Can use Weights & Biases, TrackIO
    output_dir = "outputs",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [9]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="ascii_tanks_grpo_dataset.jsonl"
)

dataset = load_dataset("json", data_files="ascii_tanks_grpo_dataset.jsonl")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [10]:
from grpo_rewards import format_reward_func, strategy_succeeds

In [11]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,  # Gives +1.0 if it outputs valid JSON with angle/power
        strategy_succeeds    # Gives up to +5.0 by instantly calculating trajectory physics offline
    ],
    
    args=training_args,
    train_dataset=dataset,
)

In [12]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 67,502,080 of 3,916,592,128 (1.72% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jovyan/.netrc.
wandb: Currently logged in as: yogeshsingla481 (yogeshsingla481-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 262144}. If this is not desired, please set these values explicitly.


Unsloth: Will smartly offload gradients to save VRAM!

[GRPO Eval] Angle: 40.0 | Power: 45.0 | Dist: 17.2 | Miss | Reward: 0.28

[GRPO Eval] Angle: 30.0 | Power: 45.0 | Dist: 16.2 | Miss | Reward: 0.38

[GRPO Eval] Angle: 30.0 | Power: 60.0 | Dist: 4.1 | 💥 HIT! Dmg: 18 | Reward: 6.80

[GRPO Eval] Angle: 60.0 | Power: 60.0 | Dist: 8.4 | Miss | Reward: 1.16


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / strategy_succeeds / mean,rewards / strategy_succeeds / std
1,0.000000,3.156911,3.120421,30.000000,26.000000,32.000000,0.000000,30.000000,26.000000,32.000000,-0.000000,1.000000,0.000000,2.156911,3.120422
2,0.000000,3.224457,6.641298,30.500000,30.000000,32.000000,0.000000,30.500000,30.000000,32.000000,-0.000000,1.000000,0.000000,2.224457,6.641298
3,0.000002,-3.446751,0.997079,29.250000,25.000000,37.000000,0.000000,29.250000,25.000000,37.000000,0.001571,0.750000,0.500000,-4.196751,1.486159
4,0.000905,0.234310,1.607461,47.250000,30.000000,71.000000,0.000000,47.250000,30.000000,71.000000,0.904836,0.750000,0.500000,-0.515690,1.159479
5,0.000002,-3.105103,0.938013,25.750000,23.000000,27.000000,0.000000,25.750000,23.000000,27.000000,0.001877,1.000000,0.000000,-4.105103,0.938013
6,0.002448,-1.408090,1.405059,55.000000,28.000000,128.000000,0.250000,30.666668,28.000000,32.000000,2.448367,1.000000,0.000000,-2.408090,1.405059
7,0.000001,0.012896,2.438463,32.250000,32.000000,33.000000,0.000000,32.250000,32.000000,33.000000,0.001019,1.000000,0.000000,-0.987104,2.438463
8,0.004488,-2.713309,0.513526,39.500000,32.000000,54.000000,0.000000,39.500000,32.000000,54.000000,4.487929,0.750000,0.500000,-3.463308,0.994610
9,0.000026,-2.895457,0.888322,25.750000,24.000000,28.000000,0.000000,25.750000,24.000000,28.000000,0.025837,1.000000,0.000000,-3.895457,0.888322
10,0.000434,-2.881269,0.880413,32.250000,27.000000,36.000000,0.000000,32.250000,27.000000,36.000000,0.434078,1.000000,0.000000,-3.881269,0.880413



[GRPO Eval] Angle: 45.0 | Power: 60.0 | Dist: 1.5 | 💥 HIT! Dmg: 44 | Reward: 9.40

[GRPO Eval] Angle: 45.0 | Power: 25.0 | Dist: 47.2 | Miss | Reward: -2.72

[GRPO Eval] Angle: 90.0 | Power: 60.0 | Dist: 60.8 | Miss | Reward: -4.08

[GRPO Eval] Angle: 60.0 | Power: 65.0 | Dist: 4.7 | 💥 HIT! Dmg: 13 | Reward: 6.30

[GRPO Eval] Angle: 180 | Power: 100 | Dist: 70.9 | Miss | Reward: -5.09

[GRPO Eval] Angle: 180 | Power: 100 | Dist: 71.3 | Miss | Reward: -5.13

[GRPO Eval] Angle: 170.0 | Power: 5.0 | Dist: 65.7 | Miss | Reward: -4.57

[GRPO Eval] Angle: 30.0 | Power: 70.0 | Dist: 11.8 | Miss | Reward: 0.82

[GRPO Eval] Angle: 60.0 | Power: 80.0 | Dist: 25.8 | Miss | Reward: -0.58

[GRPO Eval] Angle: 50.0 | Power: 75.0 | Dist: 23.0 | Miss | Reward: -0.30

[GRPO Eval] Angle: 0 | Power: 100 | Dist: 54.0 | Miss | Reward: -3.40

[GRPO Eval] Angle: 123.0 | Power: 45.0 | Dist: 74.5 | Miss | Reward: -5.45

[GRPO Eval] Angle: 90.0 | Power: 100 | Dist: 60.3 | Miss | Reward: -4.03

[GRPO Eval] Angle

profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▇▇▆▅▄▁▆▁█▆█▇▄▄▆▄▅▆▆▇▆▇▇▆▇▅▅▅▅▅▅▇▇▇▆▇██▇▇
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,▁▁▁█▁▁▂▁▁▅▁▁▃█▁▁▃▁▃▃▁▁▃▁▁▃▁▃▃▁▃▃▁▃▃▁▃▁▃▁
profiling/Time taken: UnslothGRPOTrainer.format_reward_func,▆▃▂▄▅▇▅▃▂▃█▄▄▄▄▂▄▅▂▂▃▄▄▃▂▅▂▂▃▄▂▁▁▂▂▂▂▂▂▃
profiling/Time taken: UnslothGRPOTrainer.strategy_succeeds,▅▆▅▄▅▆▁▇▁▁▇▆▄▆▃▆▅▄▅▆▇▇▅▅▆▅▅▆▆▆▆▅▅▇▆▆▅█▆▆
profiling/Time taken: UnslothGRPOTrainer.transformers.generate,█▁▂▅▁▅▅▅▆▅▁▁▃▁▄▃▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/high_max,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/high_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/low_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/low_min,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/region_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+22,...


TrainOutput(global_step=200, training_loss=0.06518484311913199, metrics={'train_runtime': 858.6266, 'train_samples_per_second': 0.233, 'train_steps_per_second': 0.233, 'total_flos': 0.0, 'train_loss': 0.06518484311913199})

In [13]:
model.save_pretrained("grpo_saved_lora")  # Local saving
tokenizer.save_pretrained("grpo_saved_lora")

['grpo_saved_lora/processor_config.json']

In [14]:
model.push_to_hub_merged(
    "yogesh1801/ministral-3b-grpo-ascii-tanks-merged",
    tokenizer,
    save_method="merged_16bit",
    token=""
)

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'max_position_embeddings'}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /home/jovyan/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:00<01:00, 60.02s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:40<00:00, 50.35s/it]


Unsloth: Merge process complete. Saved to `/home/jovyan/mistral-hackathon/yogesh1801/ministral-3b-grpo-ascii-tanks-merged`
